# 05a · 조건부 기대와 마팅게일 동물원 (Conditional Expectation and a Zoo of Martingales)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §5.1 (Conditional Expectation), §5.2 (Examples, Basic Properties) · Williams Ch. 9 (9.7 성질 목록), §10.1–10.5 · Lawler 2e §5.1–5.2 · Ross 12e Ch. 3 (§3.4 조건화 계산) · G&S 4e §7.7, §12.1 |
| 선수 노트북 | 04c (강마르코프·재생 사이클), 00c (조건부 기대의 binning 추정), 01g (골턴-왓슨 분기 과정) |
| 예상 소요 | 90분 |
| 상태 | draft |

04c에서 사슬은 상태 $x$에 도장을 찍을 때마다 "기억을 리셋"했다. 이 모듈은 기억을 리셋하는 대신 **기억을 정확히 계량하는** 도구를 배운다: 시각 $n$까지의 정보 $\mathcal F_n$과, 그 정보로 만든 최선의 예측 $E[X\mid\mathcal F_n]$. 그 위에서 **마팅게일**(martingale)은 "내일의 예측이 오늘의 값 자신인 과정", 즉 공정한 게임이다. 이 노트북은 후보 8개를 손으로 판정하고(7개는 마팅게일, 1개는 아님), 같은 판정을 조건부 평균화(binning)로 그림에서 확인한다. 05b(선택적 정지), 05d(수렴), 06d(브라운 마팅게일)가 모두 이 동물원 위에 선다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import random_walk_paths, galton_watson
from spkit.plots import plot_paths

SEED, rng = rng_for("05a")
setup_plots()
N_PATHS = scale(100_000)          # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
N_GW = scale(50_000)              # 골턴-왓슨 경로 수
MIN_BIN = 50 if fast() else 200   # 조건부 평균화에서 bin 하나의 최소 표본 수
THETA, SIGMA, GW_PMF = 0.5, 0.3, np.array([0.2, 0.3, 0.5])
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_GW={N_GW}, MIN_BIN={MIN_BIN}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다.

**Q1.** (04c) 기약·양재귀 마르코프 사슬에서 한 상태 $x$로의 복귀 시각들이 재생 과정을 이루는 이유는 무엇인가?

<details><summary>정답</summary>

강마르코프 성질: 복귀 시각 $T_x$ 직후의 미래는 과거와 독립이고 처음과 같은 분포를 가진다. 따라서 복귀 간격들이 i.i.d.이고, 복귀 시각열은 재생 과정이다.

</details>

**Q2.** (04c) $\pi(x)=1/E_x[T_x]$를 재생-보상 정리로 어떻게 증명했는가?

<details><summary>정답</summary>

각 복귀 주기를 한 사이클로 보고 사이클당 보상을 '$x$에 머문 시간($=1$)'으로 두면 장기 보상률 $=E[\text{보상}]/E[\text{주기}]=1/E_x[T_x]$. 한편 에르고딕 정리로 $x$의 장기 점유 비율은 $\pi(x)$이므로 두 값이 같다.

</details>

**Q3.** (04c) 사슬의 에르고딕 정리 $\frac1n\sum_{k<n}f(X_k)\to\sum_x\pi(x)f(x)$에 비주기성이 필요한가?

<details><summary>정답</summary>

필요 없다. 시간 평균은 주기적 사슬에서도 수렴한다(재생 구조만 필요: 기약 + 양재귀, $f$가 $\pi$-적분가능). 비주기성은 $P^n(x,y)\to\pi(y)$(분포 수렴)에만 필요하다.

</details>

**Q4.** (04b) 일반 재생 과정에서 고정 시각 $t$를 덮는 간격 길이의 극한 평균은? 푸아송이면 얼마인가?

<details><summary>정답</summary>

$E[X^2]/E[X]$ (길이 편향 표본추출). 지수(푸아송)면 $E[X^2]=2/\lambda^2$, $E[X]=1/\lambda$이므로 $2/\lambda$, 즉 보통 간격의 2배.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **필트레이션**(filtration): $\mathcal F_0\subseteq\mathcal F_1\subseteq\cdots$인 σ-대수열. 이 노트북에서는 항상 $\mathcal F_n=\sigma(X_1,\dots,X_n)$ — "시각 $n$까지 관측한 정보". 과정 $(M_n)$이 **적응**(adapted)되었다 $=$ 각 $M_n$이 $X_1,\dots,X_n$의 함수.
- **조건부 기대** $E[X\mid\mathcal F_n]$: (i) $\mathcal F_n$-가측($=X_1,\dots,X_n$의 함수)이고 (ii) 모든 $A\in\mathcal F_n$에 대해 $E[X\,\mathbf 1_A]=E[E[X\mid\mathcal F_n]\mathbf 1_A]$인 확률변수(a.s. 유일; Williams Ch. 9). 이산 경우에는 '표 $E[X\mid Y=y]$에 $Y$를 대입한 것'과 같다 — 00c에서 산점도를 bin으로 나눠 평균 낸 것이 바로 이 표의 추정이었다.
- **기본 성질** (Williams §9.7): 선형성; **탑 성질**(tower) $E[E[X\mid\mathcal F_{n+1}]\mid\mathcal F_n]=E[X\mid\mathcal F_n]$ (특히 $E[E[X\mid\mathcal F_n]]=E[X]$); **아는 것 꺼내기**(taking out what is known) $E[YX\mid\mathcal F_n]=Y\,E[X\mid\mathcal F_n]$ ($Y$가 $\mathcal F_n$-가측); **독립성** $X\perp\mathcal F_n\Rightarrow E[X\mid\mathcal F_n]=E[X]$; **젠센** $\varphi(E[X\mid\mathcal F_n])\le E[\varphi(X)\mid\mathcal F_n]$ ($\varphi$ 볼록).
- **마팅게일**: 적응 $+\ E|M_n|<\infty\ +\ E[M_{n+1}\mid\mathcal F_n]=M_n$. **서브마팅게일**(submartingale): $\ge M_n$ (평균적으로 오른다, 유리한 게임). **슈퍼마팅게일**(supermartingale): $\le M_n$ (카지노). 마팅게일 ⇔ 서브이면서 슈퍼.
- **동물원 8후보** (SRW: $X_i=\pm1$ 등확률 i.i.d., $S_n=\sum_{i\le n}X_i$, $S_0=0$):

| 후보 | 과정 | 설정 |
|---|---|---|
| Z1 | $S_n$ | SRW |
| Z2 | $S_n^2-n$ | SRW |
| Z3 | $e^{\theta S_n}/\cosh^n\theta$ | SRW, $\theta=0.5$ |
| Z4 | $Z_n/m^n$ | 골턴-왓슨, 자손 pmf $(p_0,p_1,p_2)=(0.2,0.3,0.5)$, $m=1.3$ |
| Z5 | $X_n=R_n/(R_n+B_n)$ | 폴리아 항아리, 초기 빨강 1·파랑 3, 뽑은 색 1개 추가 |
| Z6 | $L_n=\prod_{i\le n}f(X_i)/g(X_i)$ | $X_i\sim g=$Bernoulli$(1/2)$, $f=$Bernoulli$(0.6)$ → 인자 1.2(앞면)/0.8(뒷면) |
| Z7 | $\prod_{i\le n}Y_i$ | $Y_i=\exp(\sigma\xi_i-\sigma^2/2)$, $\xi_i\sim N(0,1)$, $\sigma=0.3$ |
| Z8 | $S_n^2$ | SRW — **비-예시** |

- **둡 분해**(Doob decomposition): 적응·적분가능 $X_n$은 $X_n=M_n+A_n$ ($M$ 마팅게일, $A_0=0$, $A_{n+1}-A_n=E[X_{n+1}-X_n\mid\mathcal F_n]$이 $\mathcal F_n$-가측 $=$ **예측가능**(predictable))으로 유일하게 쓰인다. 서브마팅게일 ⇔ $A_n$ 비감소.

### 2.2 정리 1 — 조건부 기대의 계산 규칙 (Williams §9.7, Durrett 3e §5.1)

$E|X|<\infty$, $\mathcal F_n\subseteq\mathcal F_{n+1}$일 때

(i) 탑 성질: $E[E[X\mid\mathcal F_{n+1}]\mid\mathcal F_n]=E[X\mid\mathcal F_n]$;
(ii) 아는 것 꺼내기: $Y$가 $\mathcal F_n$-가측이고 $E|XY|<\infty$이면 $E[XY\mid\mathcal F_n]=Y\,E[X\mid\mathcal F_n]$;
(iii) 독립성: $X$가 $\mathcal F_n$과 독립이면 $E[X\mid\mathcal F_n]=E[X]$.

가정이 왜 필요한가:
- **$E|X|<\infty$**: 조건부 기대가 정의되려면 적분가능해야 한다(코시 분포 같은 경우 정의 불가).
- **$Y$의 $\mathcal F_n$-가측성**: '아는 것'만 꺼낼 수 있다. $Y=X_{n+1}$(미래)을 꺼내면 안 된다 — $E[X_{n+1}S_n\mid\mathcal F_n]=S_nE[X_{n+1}]$이 되는 이유는 $S_n$을 (ii)로 꺼내고 $X_{n+1}$에 (iii)을 쓴 것이지, $X_{n+1}$을 꺼낸 것이 아니다. 방향을 혼동하기 쉽다.
- **(iii)의 독립성**은 σ-대수 전체와의 독립을 뜻한다. $X$가 각 $X_i$와만 독립(쌍독립)이면 부족하다.

*증명 스케치.* 이산 경우($\mathcal F_n$이 유한 분할 $\{A\}$로 생성)에는 $E[X\mid\mathcal F_n]=\sum_A\mathbf 1_A\,E[X\mathbf 1_A]/P(A)$이다. (i): 안쪽 조건부 기대는 더 잘게 나눈 분할 $\{A'\}$의 bin 평균이고, 바깥은 그것을 다시 큰 bin $A$ 안에서 $P(A'\mid A)$ 가중 평균하는 것 — 잘게 나눈 평균의 평균은 처음부터 크게 나눈 평균과 같다. (ii): $Y$가 각 $A$ 위에서 상수 $y_A$이므로 $E[XY\mathbf 1_A]=y_AE[X\mathbf 1_A]$. (iii): $E[X\mathbf 1_A]=E[X]P(A)$이므로 모든 bin에서 평균이 $E[X]$. 일반 경우는 각 성질이 정의 (ii)의 특성 방정식 $E[X\mathbf 1_A]=E[\,\cdot\,\mathbf 1_A]$를 만족함을 확인하는 것으로 환원된다 — Williams §9.7. $\square$

### 2.3 정리 2 — 동물원 판정 정리 (Durrett 3e §5.2, Lawler 2e §5.2)

정의된 필트레이션 아래에서 **Z1–Z7은 마팅게일**이다. **Z8** $S_n^2$는 $E[S_{n+1}^2\mid\mathcal F_n]=S_n^2+1>S_n^2$이므로 **서브마팅게일이며 마팅게일이 아니다.** 마팅게일이면 모든 $n$에서 $E[M_n]=E[M_0]$.

가정이 왜 필요한가:
- **Z1**: 증분 평균 0 ($E[X]=0$). $p\ne1/2$이면 서브/슈퍼마팅게일 (E3).
- **Z2**: 증분 분산 1 ($E[X^2]=1$). 일반적으로 $S_n^2-n\sigma^2$.
- **Z3**: $E[e^{\theta X}]=\cosh\theta<\infty$ — 지수 모멘트 존재. 정규화 없이는 젠센에 의해 서브마팅게일.
- **Z4**: 자손 평균 $m<\infty$; 필트레이션은 $\sigma(Z_0,\dots,Z_n)$ (또는 각 개체의 자손 수까지 포함해도 됨).
- **Z5**: '뽑은 색과 같은 색 한 개 추가' 규칙. 반대 색을 추가하면 마팅게일이 아니다(05d E3).
- **Z6**: 관측 $X_i\sim g$이고 $f>0$인 곳에서 $g>0$. $f$ 아래에서는 $1/L_n$이 마팅게일이다(반대 방향 $g>0\Rightarrow f>0$도 성립할 때 — $E_f[g/f]=\sum_{f>0}g=1$이 되려면 필요하다; 예제에서는 둘 다 성립).
- **Z7**: 독립 + 평균 1 + 적분가능. 평균이 1이 아니면 기하적으로 발산/소멸.

<details><summary>증명</summary>

모든 후보가 적응·적분가능임은 명백하다(유한 합·곱, 유계 또는 유한 모멘트). 각 후보에 대해 $E[M_{n+1}\mid\mathcal F_n]$을 한 줄로 계산한다. 아래에서 $X=X_{n+1}$은 $\mathcal F_n$과 독립이므로 정리 1 (iii)을, $S_n$은 $\mathcal F_n$-가측이므로 (ii)를 쓴다.

**Z1.** $E[S_n+X\mid\mathcal F_n]=S_n+E[X]=S_n+0$.

**Z2.** $E[(S_n+X)^2\mid\mathcal F_n]=S_n^2+2S_nE[X]+E[X^2]=S_n^2+1$. 따라서 $E[S_{n+1}^2-(n+1)\mid\mathcal F_n]=S_n^2+1-(n+1)=S_n^2-n$.

**Z3.** $E[e^{\theta S_{n+1}}\mid\mathcal F_n]=e^{\theta S_n}E[e^{\theta X}]=e^{\theta S_n}\cosh\theta$. $\cosh^{n+1}\theta$로 나누면 $e^{\theta S_n}/\cosh^n\theta$.

**Z4.** $Z_{n+1}=\sum_{j=1}^{Z_n}\xi_j^{(n+1)}$, 자손 수 $\xi_j^{(n+1)}$은 $\mathcal F_n$과 독립·평균 $m$. 조건부로 $Z_n$은 상수이므로 $E[Z_{n+1}\mid\mathcal F_n]=mZ_n$, 즉 $E[Z_{n+1}/m^{n+1}\mid\mathcal F_n]=Z_n/m^n$.

**Z5.** 시각 $n$에 빨강 $R$, 전체 $N=R+B$라 하자. 빨강을 뽑을 확률 $R/N$이면 다음 비율은 $(R+1)/(N+1)$, 아니면 $R/(N+1)$:
$$E[X_{n+1}\mid\mathcal F_n]=\frac{\frac RN(R+1)+\big(1-\frac RN\big)R}{N+1}=\frac{R(R+1)+(N-R)R}{N(N+1)}=\frac{R(N+1)}{N(N+1)}=\frac RN=X_n.$$

**Z6.** $E_g[f(X)/g(X)]=\sum_{x:g(x)>0}g(x)\frac{f(x)}{g(x)}=\sum_xf(x)=1$ ($f>0\Rightarrow g>0$이므로 합에서 빠지는 $x$가 없다). 아는 것 꺼내기로 $E[L_{n+1}\mid\mathcal F_n]=L_n\,E[f(X_{n+1})/g(X_{n+1})]=L_n$. 예제에서 $\frac12\cdot1.2+\frac12\cdot0.8=1$.

**Z7.** 같은 논리: $E[Y_{n+1}]=e^{-\sigma^2/2}E[e^{\sigma\xi}]=e^{-\sigma^2/2}e^{\sigma^2/2}=1$이므로 $E[\prod_{i\le n+1}Y_i\mid\mathcal F_n]=\prod_{i\le n}Y_i$.

**Z8.** Z2 계산의 부산물: $E[S_{n+1}^2\mid\mathcal F_n]=S_n^2+1\ge S_n^2$, 등호가 성립하지 않으므로 서브마팅게일이되 마팅게일이 아니다.

**기대값 보존.** $E[M_{n+1}]=E[E[M_{n+1}\mid\mathcal F_n]]=E[M_n]$ (탑 성질), 귀납으로 $E[M_n]=E[M_0]$. 서브마팅게일이면 같은 계산이 $E[M_{n+1}]\ge E[M_n]$을 준다: Z8에서 $E[S_n^2]=n$. $\square$

</details>

### 2.4 정리 3 — 둡 분해 (Doob decomposition)

적응·적분가능 과정 $X_n$에 대해 $A_0=0$, $A_{n+1}=A_n+E[X_{n+1}-X_n\mid\mathcal F_n]$, $M_n=X_n-A_n$으로 두면 $M$은 마팅게일, $A$는 예측가능($A_{n+1}$이 $\mathcal F_n$-가측)이며 이런 분해는 유일하다. $X$가 서브마팅게일 ⇔ $A$ 비감소.

가정이 왜 필요한가:
- **예측가능성**(한 시각 앞서 알 수 있음)이 유일성의 핵심: $A$를 단지 적응으로만 요구하면 $X=X+0$ 등 무수한 분해가 생긴다.
- **적분가능성**은 조건부 기대 정의에 필요.

*증명 스케치.* $E[M_{n+1}-M_n\mid\mathcal F_n]=E[X_{n+1}-X_n\mid\mathcal F_n]-(A_{n+1}-A_n)=0$이고, $A_{n+1}$은 $\mathcal F_n$-가측인 것들의 합이라 예측가능. 유일성: 두 분해의 차 $M-M'=A'-A$는 예측가능 마팅게일 → $A'_{n+1}-A_{n+1}=E[A'_{n+1}-A_{n+1}\mid\mathcal F_n]=A'_n-A_n=\cdots=A'_0-A_0=0$. 예: Z8 $S_n^2=(S_n^2-n)+n$, $A_n=n$ — 증가하는 예측가능 부분이 정확히 '드리프트 $+1$'이다. $\square$

### 2.5 직관

조건부 기대는 '지금까지 본 것으로 만든 최선의 예측'이고, 마팅게일은 '내일의 예측이 오늘의 값 자신인 과정' $=$ 공정한 게임이다. 동물원의 공통 패턴:

- **함수형** $h(S_n,n)$이 마팅게일이려면 이산 열방정식 $\tfrac12[h(s+1,n+1)+h(s-1,n+1)]=h(s,n)$을 만족해야 한다. $s$, $s^2-n$, $s^3-3ns$ (E1), $e^{\theta s}\cosh^{-n}\theta$가 그 해들이다 — 06d의 브라운 마팅게일 $B_t$, $B_t^2-t$, $e^{\theta B_t-\theta^2t/2}$의 이산판.
- **곱형** (Z6, Z7)은 '평균 1인 인자를 곱해 나가는' 구조. 골턴-왓슨 Z4는 '평균 $m$으로 불어나는 것을 $m^n$으로 나눈' 정규화이다.
- **항아리형** (Z5)은 강화(reinforcement)가 정확히 비율을 보존하도록 맞춰져 있다.

**수치 검증의 원리.** $M_n$이 마르코프 상태의 함수이면 $E[M_{n+1}\mid\mathcal F_n]=E[M_{n+1}\mid\text{상태}_n]$이므로, 상태별로 bin을 나눠 $M_{n+1}$을 평균 내면 (bin의 $M_n$ 값에 대해) 대각선 $y=x$ 위에 놓여야 한다 — 00c의 binning 그림과 같은 방법이다. 비-예시 Z8은 대각선에서 $+1$만큼 벗어난다.

### 2.6 함정 (traps)

- '모든 $n$에서 $E[M_n]=M_0$'은 마팅게일의 **필요조건일 뿐 충분조건이 아니다.** 예: 독립 잡음 $M_n=\xi_n$ (평균 0)은 $E[M_n]=0$이지만 $E[M_{n+1}\mid\mathcal F_n]=0\ne M_n$. 4.4의 표만 보고 판정하면 안 되고 4.3의 조건부 그림이 필요하다.
- 마팅게일 여부는 **필트레이션에 의존**한다. $M_{n+1}$ 자체를 이미 아는 σ-대수(예: $\mathcal G_n=\mathcal F_{n+1}$)에 대해서는 $E[M_{n+1}\mid\mathcal G_n]=M_{n+1}$이므로 경로가 상수가 아닌 한 마팅게일이 아니다. 단, '정보가 더 많다'는 것 자체가 문제는 아니다 — 과정과 독립인 $Y$를 덧붙인 $\mathcal G_n=\sigma(X_1,\dots,X_n,Y)$에 대해서는 $E[X_{n+1}\mid\mathcal G_n]=0$이라 $S_n$이 여전히 마팅게일이다. 깨뜨리는 것은 증분 $M_{n+1}-M_n$에 대한 정보다. 반대로 더 작은 필트레이션으로 바꿀 때는 $M_n$이 그 필트레이션에 적응되어 있어야 한다.
- 조건부 평균화로 $M_n$ 값만으로 bin을 나누는 것은 $M_n$이 **마르코프 상태의 함수일 때만** $\mathcal F_n$ 조건화와 같다(Z1–Z8은 모두 그렇다). 일반 과정에서는 경로 전체에 조건을 걸어야 한다.
- 폴리아 항아리의 $E[X_n]=a/(a+b)$는 시간에 무관하지만 개별 경로는 초기 몇 번의 뽑기에 크게 좌우된다. '평균이 일정'과 '값이 일정한 상수로 수렴'은 다르다(05d).
- $e^{\theta S_n}$ 자체는 젠센에 의해 서브마팅게일($\cosh\theta\ge1$)이다. 정규화 $\cosh^{-n}\theta$를 잊으면 판정이 뒤바뀐다.
- 우도비 Z6에서 **어느 분포 아래에서** 기대를 취하는지(관측이 $g$에서 나옴)를 명시하지 않으면 마팅게일 여부를 말할 수 없다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — '마팅게일'을 도박·예측·보존이라는 세 단어를 모두 써서 두 문장으로 설명하고, Z8이 왜 실패하는지 한 줄로.)

## 3. Predict — 코드를 돌리기 전에 예측

코드를 돌리기 전에 아래 값을 먼저 적어 보세요. 손계산 힌트: 정리 2의 각 후보에 대해 $E[M_n]=M_0$, 그리고 Z8의 조건부 계산 $E[S_{n+1}^2\mid S_n=s]=s^2+1$.

In [ ]:
predictions = {
    # E[S_10^2]은? (SRW, S_0 = 0)
    "E_S10_sq": None,
    # E[S_11^2 | S_10 = 2]는? (비-예시 Z8의 드리프트)
    "cond_S11sq_given_S10_2": None,
    # E[exp(0.5 S_10)]은? (Z3의 정규화 상수를 생각해 보세요)
    "exp_mart_mean": None,
    # 자손 pmf (0.2, 0.3, 0.5), Z_0 = 1인 골턴-왓슨의 E[Z_5]는?
    "gw_mean_Z5": None,
    # 빨강 1·파랑 3으로 시작한 폴리아 항아리에서 20번 뽑은 뒤 빨간 비율의 기대값 E[X_20]은?
    "polya_mean_X20": None,
    # 공정 동전 10번의 관측으로 만든 우도비 L_10 = 1.2^{#H} 0.8^{#T}의 기대값(공정 동전 아래)은?
    "lr_mean_L10": None,
}

## 4. Simulate — 시뮬레이션

먼저 이 노트북에서만 쓰는 도우미를 정의한다. 항아리·우도비·곱형 경로 생성기 세 개와, 00c의 binning 추정을 함수로 묶은 `binned_conditional_mean`이다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def polya_urn(a, b, n_draws, rng, n_paths):
    """폴리아 항아리: 빨강 a·파랑 b로 시작, 뽑은 색 1개 추가. 빨간 비율 X_n, shape (n_paths, n_draws+1)."""
    red, total = np.full(n_paths, a, dtype=float), float(a + b)
    X = np.empty((n_paths, n_draws + 1)); X[:, 0] = red / total
    for k in range(n_draws):
        red += rng.random(n_paths) < red / total     # 빨강을 뽑으면 빨강 +1
        total += 1.0
        X[:, k + 1] = red / total
    return X

def coin_lr_paths(n, rng, n_paths, p_f=0.6):
    """관측 X_i ~ g = Bernoulli(1/2), 대립 f = Bernoulli(p_f). 우도비 L_k (k=0..n)와 앞면 행렬."""
    heads = rng.random((n_paths, n)) < 0.5
    factors = np.where(heads, p_f / 0.5, (1 - p_f) / 0.5)
    L = np.ones((n_paths, n + 1)); L[:, 1:] = np.cumprod(factors, axis=1)
    return L, heads

def lognormal_product_paths(n, sigma, rng, n_paths):
    """Y_i = exp(sigma xi_i - sigma^2/2) (평균 1)의 누적곱, shape (n_paths, n+1)."""
    Y = np.exp(sigma * rng.standard_normal((n_paths, n)) - sigma**2 / 2)
    M = np.ones((n_paths, n + 1)); M[:, 1:] = np.cumprod(Y, axis=1)
    return M

def binned_conditional_mean(x, y, min_count=200, n_quantile_bins=None):
    """E[y | x in bin]: 이산 x는 값별로, 연속 x는 분위수 bin으로. (bin의 x 평균, y 평균, y SE, 개수)."""
    if n_quantile_bins is None:
        keys = np.unique(x); idx = np.searchsorted(keys, x)
    else:
        edges = np.quantile(x, np.linspace(0, 1, n_quantile_bins + 1))
        idx = np.clip(np.searchsorted(edges, x, side="right") - 1, 0, n_quantile_bins - 1)
    rows = []
    for j in np.unique(idx):
        m = idx == j
        if m.sum() >= min_count:
            r = mc_estimate(y[m]); rows.append((x[m].mean(), r.mean, r.se, m.sum()))
    if not rows:
        raise ValueError(f"no bin has >= {min_count} samples")
    return tuple(np.array(c) for c in zip(*rows))

### 4.1 조건부 기대 복습: $E[S_{11}^2\mid S_{10}=s]$ (Fig 1)

SRW 경로를 만들고 $S_{10}=s$인 경로들만 모아 $S_{11}^2$의 평균을 낸다($s\in\{-10,-8,\dots,10\}$; FAST 모드에서는 $\pm10$ bin의 경로가 몇 개 없어 빠질 수 있다). 정리 2의 Z8 계산은 $s^2+1$을 예언하고, 마팅게일이었다면 $s^2$이었을 것이다. 같은 bin에서 Z2 $S_{11}^2-11$의 평균은 $s^2-10$, 즉 bin의 $M_{10}$ 값과 같아야 한다.

In [ ]:
paths = random_walk_paths(N_PATHS, 20, rng)          # shape (N_PATHS, 21), S_0 = 0
S10, S11 = paths[:, 10], paths[:, 11]

MIN_BIN_FIG1 = 20 if not fast() else 5                 # s = ±10은 2^-10 비율이라 이 그림만 문턱을 낮춘다
bin_s, cm_S11sq, se_S11sq, cnt = binned_conditional_mean(S10, S11**2, MIN_BIN_FIG1)
_, cm_Z2, se_Z2, _ = binned_conditional_mean(S10, S11**2 - 11, MIN_BIN_FIG1)
print("s          :", bin_s.astype(int))
print("count      :", cnt)
print("E[S11^2|s] :", cm_S11sq.round(2), "  vs s^2+1 :", (bin_s**2 + 1).astype(int))
print("E[Z2_11|s] :", cm_Z2.round(2), "  vs s^2-10:", (bin_s**2 - 10).astype(int))
p_S10_eq_2 = mc_proportion(S10 == 2)
print(f"P(S_10 = 2) = {p_S10_eq_2}  (exact C(10,6)/2^10 = {210/1024:.6f})")

In [ ]:
fig, ax = plt.subplots()
s_grid = np.linspace(bin_s.min() - 0.5, bin_s.max() + 0.5, 200)
ax.plot(s_grid, s_grid**2 + 1, color="C1", lw=1.5, label="$s^2 + 1$ (Z8: submartingale drift)")
ax.plot(s_grid, s_grid**2, color="k", ls="--", lw=1, label="$s^2$ (what a martingale would give)")
ax.errorbar(bin_s, cm_S11sq, yerr=1.96 * se_S11sq, fmt="o", color="C0", capsize=3,
            label=f"MC bin mean, 95% CI ({N_PATHS} paths)")
ax.set_xlabel("$s = S_{10}$"); ax.set_ylabel("$E[S_{11}^2 \\mid S_{10} = s]$")
ax.set_title("Fig 1. Conditional mean $E[S_{11}^2 \\mid S_{10} = s]$ vs $s$"); ax.legend()
plt.show()

점들은 $s^2+1$ 위에 놓이고 $s^2$ 위에는 놓이지 않는다 — 두 곡선의 차이가 정확히 1이라 그림에서는 작아 보이지만, $s=\pm2$ bin의 CI 폭(≈0.1)에 비하면 10배 이상이다($s=0$ bin은 $S_{11}^2=1$로 결정적이라 CI 폭이 0이다). 양 끝 $s=\pm10$ bin은 경로가 $2^{-10}$ 비율로 드물어 오차 막대가 크다. 조건부 기대는 "bin별 평균"이라는 것, 그리고 $S_n^2$는 매 걸음 평균 1씩 자란다는 것을 동시에 보여 준다.

### 4.2 동물원 생성

후보마다 (bin 변수, $M_n$, $M_{n+1}$) 삼중을 만든다. Z1·Z2·Z3·Z8은 4.1의 경로를 재사용($n=10\to11$), Z4는 세대 $5\to6$, Z5는 뽑기 $20\to21$, Z6·Z7은 $10\to11$. bin 변수는 마르코프 상태다: SRW 위치 $S_{10}$, 개체 수 $Z_5$, 빨간 공 수 $R_{20}$, 앞면 수, 그리고 Z7은 $M_{10}$ 자체(연속이라 분위수 bin).

In [ ]:
Z = galton_watson(GW_PMF, 6, rng, n_paths=N_GW)                 # shape (N_GW, 7)
m_gw = GW_PMF @ np.arange(3)                                     # 자손 평균 m = 1.3
Xp = polya_urn(1, 3, 21, rng, N_PATHS)                           # shape (N_PATHS, 22)
L, heads = coin_lr_paths(11, rng, N_PATHS)                       # L shape (N_PATHS, 12)
Mln = lognormal_product_paths(11, SIGMA, rng, N_PATHS)           # shape (N_PATHS, 12)

exp_mart = lambda S, n: np.exp(THETA * S) / np.cosh(THETA)**n
zoo = {   # panel title: (bin variable, M_n, M_{n+1}, M_0, plain-text label for tables)
    "Z1: $S_n$":                  (S10, S10, S11, 0.0, "Z1: S_n"),
    "Z2: $S_n^2 - n$":            (S10, S10**2 - 10, S11**2 - 11, 0.0, "Z2: S_n^2 - n"),
    "Z3: $e^{\\theta S_n}/\\cosh^n\\theta$": (S10, exp_mart(S10, 10), exp_mart(S11, 11), 1.0, "Z3: exp(theta S_n)/cosh(theta)^n"),
    "Z4: $Z_n/m^n$ (GW)":         (Z[:, 5], Z[:, 5] / m_gw**5, Z[:, 6] / m_gw**6, 1.0, "Z4: Z_n/m^n (GW)"),
    "Z5: Polya red fraction":     (np.rint(Xp[:, 20] * 24), Xp[:, 20], Xp[:, 21], 0.25, "Z5: Polya red fraction"),
    "Z6: likelihood ratio $L_n$": (heads[:, :10].sum(1), L[:, 10], L[:, 11], 1.0, "Z6: likelihood ratio L_n"),
    "Z7: $\\prod Y_i$ (mean 1)":  (Mln[:, 10], Mln[:, 10], Mln[:, 11], 1.0, "Z7: prod Y_i (mean 1)"),
    "Z8: $S_n^2$ (NOT a martingale)": (S10, S10**2, S11**2, 0.0, "Z8: S_n^2 (NOT a martingale)"),
}
for name, (b, Mn, Mn1, M0, label) in zoo.items():
    print(f"{label:36s} M_0={M0:5.2f}  mean M_n={Mn.mean():8.4f}  mean M_n+1={Mn1.mean():8.4f}")

### 4.3 조건부 평균화 검증 (Fig 2)

각 후보에 대해 bin별 $E[M_{n+1}\mid\text{bin}]$을 그 bin의 $M_n$ 평균에 대해 그린다. 마팅게일이면 점들이 대각선 $y=x$ 위에, Z8은 $y=x+1$ 위에 놓여야 한다. Z7만 연속 변수라 20-분위 bin을 쓴다 — 마팅게일 성질은 $\mathcal F_{10}$의 임의 사건에 조건을 걸어도 성립하므로 bin 안 평균끼리 비교해도 정확하다.

In [ ]:
binned = {}
for name, (b, Mn, Mn1, M0, label) in zoo.items():
    q = 20 if name.startswith("Z7") else None
    bx, by, bse, bc = binned_conditional_mean(b, Mn1, MIN_BIN, q)
    bmn = binned_conditional_mean(b, Mn, MIN_BIN, q)[1]          # 같은 bin 안 M_n 평균 (x축)
    binned[name] = (bmn, by, bse, bc)
    z = np.abs(by - bmn) / np.where(bse > 0, bse, np.inf)      # 결정적 bin(SE=0)은 z=0으로 처리
    print(f"{label:36s} {len(bx):2d} bins, max |E[M_n+1|bin] - M_n| / SE = {z.max():5.2f}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, (name, (bmn, by, bse, bc)) in zip(axes.ravel(), binned.items()):
    lo, hi = bmn.min(), bmn.max(); pad = 0.08 * (hi - lo) + 1e-9
    d = np.array([lo - pad, hi + pad])
    ax.plot(d, d, "k--", lw=1, label="$y = x$")
    if name.startswith("Z8"):
        ax.plot(d, d + 1, color="C1", lw=1.5, label="$y = x + 1$")
    ax.errorbar(bmn, by, yerr=1.96 * bse, fmt="o", ms=4, color="C0", capsize=2, label="MC bin mean, 95% CI")
    dev = np.average(by - bmn, weights=bc)                    # 대각선에서의 (표본 가중) 평균 편차
    ax.text(0.97, 0.05, f"mean deviation from y = x: {dev:+.3f}", transform=ax.transAxes,
            ha="right", va="bottom", fontsize=8, bbox=dict(boxstyle="round", fc="white", alpha=0.8))
    ax.set_title(name, fontsize=10); ax.set_xlabel("$M_n$ (bin mean)"); ax.set_ylabel("$E[M_{n+1} \\mid$ bin$]$")
    ax.legend(fontsize=7, loc="upper left")
fig.suptitle("Fig 2. $E[M_{n+1} \\mid M_n]$ vs $M_n$ for the eight candidates", y=1.0)
plt.tight_layout(); plt.show()

Z1–Z7의 점들은 거의 모두 오차 막대 안에서 대각선에 붙어 있다(95% CI이므로 20개 중 1개꼴로 벗어나는 것이 정상이다 — 위 출력의 max |z|가 2–3 정도인 것도 그 때문). Z8은 대각선에서 정확히 1만큼 위로 평행이동한 직선 위에 있다 — 둡 분해 $S_n^2=(S_n^2-n)+n$의 예측가능 증분 $A_{n+1}-A_n=1$이 그대로 보인다. Z5(폴리아)에서 오른쪽 끝의 오차 막대가 크고 bin이 잘리는 것은(표본 수 `MIN_BIN` 미만인 큰 $R_{20}$ 값은 버린다) 빨간 공이 많이 뽑힌 경로가 드물기 때문이다(초기 빨강 1).

### 4.4 전역 검증 $E[M_n]=M_0$

정리 2의 마지막 문장: 마팅게일이면 고정 시각의 기대값이 보존된다. 8후보의 표본 평균을 $M_0$과 비교한다. Z8은 $E[S_{10}^2]=10\ne0$이라 표에서 실패가 드러나야 한다 — 그리고 이 표는 함정 1(필요조건일 뿐)을 잊지 말고 4.3과 함께 읽어야 한다.

In [ ]:
rows_zoo = [{"name": label, "predicted": None, "estimate": mc_estimate(Mn), "exact": M0}
            for name, (b, Mn, Mn1, M0, label) in zoo.items()]
Markdown(compare_table(rows_zoo))

### 4.5 05d 예고: 폴리아 항아리의 경로들 (Fig 3)

$E[X_n]=1/4$는 모든 $n$에서 같다(정리 2). 그런데 경로 하나하나는 어디로 가는가? 12개 경로를 300번 뽑기까지 그려 본다.

In [ ]:
Xp_long = polya_urn(1, 3, 300, rng, 12)
fig, ax = plt.subplots()
plot_paths(np.arange(301), Xp_long, ax=ax, n_show=12, alpha=0.8, lw=1)
ax.axhline(0.25, color="k", ls="--", lw=1.2, label="$E[X_n] = 1/4$ for every $n$")
ax.set_xlabel("n (draws)"); ax.set_ylabel("fraction of red $X_n$"); ax.set_ylim(0, 1)
ax.set_title("Fig 3. Polya urn (1 red, 3 blue): fraction of red vs n, 12 paths"); ax.legend()
plt.show()
print("X_300 of the 12 paths:", Xp_long[:, -1].round(3))

각 경로는 초기 몇 번의 뽑기에서 결정된 서로 다른 값 근처에 정착한다. 평균은 0.25로 일정하지만 "0.25로 수렴하는 경로"는 없다 — 극한이 Beta(1,3) 확률변수라는 사실과 그 증명(마팅게일 수렴 정리)은 05d에서 다룬다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 모두 코드에서 계산한다: 정리 2의 $E[M_n]=M_0$, Z8의 $E[S_{n+1}^2\mid S_n=s]=s^2+1$, 그리고 정규화 상수 $\cosh^{10}\theta$, $m^5$.

In [ ]:
n0 = 10
exact = {
    "E_S10_sq": float(n0),                                   # E[S_n^2 - n] = 0  =>  E[S_n^2] = n
    "cond_S11sq_given_S10_2": 2.0**2 + 1,                    # s^2 + 1
    "cond_Z2_given_S10_2": 2.0**2 - n0,                      # (s^2 - n) 보존
    "exp_mart_mean": np.cosh(THETA)**n0,                     # E[e^{theta S_n}] = cosh^n theta
    "gw_mean_Z5": m_gw**5,                                   # E[Z_n] = m^n
    "polya_mean_X20": 1 / (1 + 3),                           # X_0 = a/(a+b)
    "lr_mean_L10": 1.0,                                      # L_0 = 1
    "lognormal_M10": 1.0,                                    # M_0 = 1
}
est_S10sq = mc_estimate(S10.astype(float)**2)
est_cond_Z8 = mc_estimate((S11**2)[S10 == 2].astype(float))
est_cond_Z2 = mc_estimate((S11**2 - 11)[S10 == 2].astype(float))
est_Z3 = mc_estimate(exp_mart(S10, n0))              # E[e^{theta S_10}] / cosh^10 theta -> 1
est_Z4 = mc_estimate(Z[:, 5] / m_gw**5)              # E[Z_5] / m^5 -> 1
est_Z5 = mc_estimate(Xp[:, 20])
est_Z6 = mc_estimate(L[:, 10])
est_Z7 = mc_estimate(Mln[:, 10])
print(f"exact: cosh(0.5)^10 = {exact['exp_mart_mean']:.6f},  m^5 = {exact['gw_mean_Z5']:.6f}")

Z3·Z4 행은 비율 추정/정규화 상수를 1과 비교한다(예측값도 같은 상수로 나눈다). 다른 행은 값 그대로. 행 이름의 `\|`는 Markdown 표 안에서 조건 기호 `|`를 셀 구분자와 구별하기 위한 이스케이프다.

In [ ]:
rows = [
    {"name": "E[S_10^2] (Z2)", "predicted": predictions["E_S10_sq"], "estimate": est_S10sq, "exact": exact["E_S10_sq"]},
    {"name": "E[S_11^2 - 11 \\| S_10 = 2] (Z2, conditional)", "predicted": None, "estimate": est_cond_Z2, "exact": exact["cond_Z2_given_S10_2"]},
    {"name": "E[S_11^2 \\| S_10 = 2] (Z8, non-example)", "predicted": predictions["cond_S11sq_given_S10_2"], "estimate": est_cond_Z8, "exact": exact["cond_S11sq_given_S10_2"]},
    {"name": "E[exp(0.5 S_10)] / cosh(0.5)^10 (Z3)", "predicted": None if predictions["exp_mart_mean"] is None else predictions["exp_mart_mean"] / exact["exp_mart_mean"], "estimate": est_Z3, "exact": 1.0},
    {"name": "E[Z_5] / 1.3^5 (Z4)", "predicted": None if predictions["gw_mean_Z5"] is None else predictions["gw_mean_Z5"] / exact["gw_mean_Z5"], "estimate": est_Z4, "exact": 1.0},
    {"name": "E[X_20] Polya(1,3) (Z5)", "predicted": predictions["polya_mean_X20"], "estimate": est_Z5, "exact": exact["polya_mean_X20"]},
    {"name": "E_g[L_10] coin likelihood ratio (Z6)", "predicted": predictions["lr_mean_L10"], "estimate": est_Z6, "exact": exact["lr_mean_L10"]},
    {"name": "E[prod_{i<=10} Y_i] lognormal (Z7)", "predicted": None, "estimate": est_Z7, "exact": exact["lognormal_M10"]},
]
Markdown(compare_table(rows))

각 행이 어긋난다면 무엇이 의심되는가:
- **E[S_10^2]**: Z2가 마팅게일이 아니거나(증분 분산 ≠ 1), `random_walk_paths`의 걸음이 $\pm1$이 아니다.
- **E[S_11^2 − 11 | S_10 = 2]**: bin 마스크 `S10 == 2`가 시각을 잘못 잡았다(열 인덱스 10이 $S_{10}$인지 확인).
- **E[S_11^2 | S_10 = 2]**: 드리프트가 $+1$이 아니면 $E[X^2]\ne1$ — 걸음 크기 문제.
- **Z3**: 정규화 상수 $\cosh^{10}\theta$를 빠뜨렸거나 $\theta$가 다르다(SE ≈ 0.008 full, 꼬리가 무거워 FAST에서는 0.025).
- **Z4**: `galton_watson`의 자손 평균이 1.3이 아니거나 세대 인덱스가 어긋났다.
- **Z5**: 항아리 규칙(뽑은 색 추가)이 잘못 구현되었다 — 반대 색을 추가하면 평균이 1/2로 끌려간다.
- **Z6**: 관측 동전이 공정하지 않거나(기대를 $f$ 아래에서 취함) 인자 1.2/0.8이 뒤바뀌었다.
- **Z7**: 보정항 $-\sigma^2/2$를 빠뜨렸다 — 그러면 평균이 $e^{n\sigma^2/2}$로 자란다.

In [ ]:
assert_close(est_S10sq, exact["E_S10_sq"], k=4, name="E[S_10^2]")
assert_close(est_cond_Z2, exact["cond_Z2_given_S10_2"], k=4, name="E[S_11^2-11 | S_10=2]")
assert_close(est_cond_Z8, exact["cond_S11sq_given_S10_2"], k=4, name="E[S_11^2 | S_10=2] (non-example drift)")
assert_close(est_Z3, 1.0, k=4, name="exp martingale")
assert_close(est_Z4, 1.0, k=4, name="GW Z_5/m^5")
assert_close(est_Z5, exact["polya_mean_X20"], k=4, name="Polya(1,3) E[X_20]")
assert_close(est_Z6, exact["lr_mean_L10"], k=4, name="coin likelihood ratio")
assert_close(est_Z7, exact["lognormal_M10"], k=4, name="product of iid mean-1")
print("all assert_close passed (k=4)")

## 6. 연습문제

### E1 계산

(a) $E[S_{n+1}^3\mid\mathcal F_n]$을 계산하여 $S_n^3$이 마팅게일이 아님을 보여라. (b) $S_n^3-3nS_n$이 마팅게일임을 보여라. (c) Z6에서 임의의 이산 분포 $f,g$ ($f>0\Rightarrow g>0$)에 대해 $E_g[f(X)/g(X)]=1$을 증명하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $X=X_{n+1}$은 $\mathcal F_n$과 독립이고 $E[X]=E[X^3]=0$, $E[X^2]=1$. 이항 전개 후 $S_n$의 거듭제곱을 꺼내면
$E[(S_n+X)^3\mid\mathcal F_n]=S_n^3+3S_n^2E[X]+3S_nE[X^2]+E[X^3]=S_n^3+3S_n\ne S_n^3$ ($S_n\ne0$일 때). 서브도 슈퍼도 아니다 — $S_n$의 부호에 따라 드리프트 방향이 바뀐다.

(b) $E[S_{n+1}^3-3(n+1)S_{n+1}\mid\mathcal F_n]=(S_n^3+3S_n)-3(n+1)S_n=S_n^3-3nS_n$. 열방정식 $\frac12[h(s+1,n+1)+h(s-1,n+1)]=h(s,n)$의 세 번째 다항식 해 $h=s^3-3ns$이다. 따라서 $E[S_{10}^3-30S_{10}]=0$.

(c) $E_g[f(X)/g(X)]=\sum_{x:g(x)>0}g(x)\frac{f(x)}{g(x)}=\sum_{x:g(x)>0}f(x)=\sum_xf(x)=1$ — 마지막 등호는 $g(x)=0\Rightarrow f(x)=0$이라 빠진 항이 0이기 때문. 이 조건이 깨지면(어떤 $x$에서 $f>0=g$) 합이 1보다 작아 $L_n$은 슈퍼마팅게일이 된다.

</details>

### E2 유도 후 시뮬레이션 검증

E1(b)의 마팅게일 $M_n=S_n^3-3nS_n$을 검증하라: $E[S_{10}^3-30S_{10}]=0$, 조건부 평균화 그림(bin: $S_{10}$), 그리고 $E[S_{11}^3-33S_{11}\mid S_{10}=2]$. 손으로 먼저: 마지막 값은 얼마여야 하는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$E[M_{11}\mid S_{10}=2]=M_{10}=2^3-30\cdot2=-52$ (직접 계산해도 $\frac12[(27-99)+(1-33)]=-52$). $M_{10}$의 표준편차는 약 66.0이므로 100k 경로에서 SE ≈ 0.21; bin $S_{10}=2$ 안에서는 두 값 $-72,-32$ 중 하나라 SD $=20$, SE ≈ $20/\sqrt{20500}\approx0.14$.

```python
S = random_walk_paths(N_PATHS, 11, rng); n = np.arange(12)
M3 = S**3 - 3 * n * S                                        # shape (N_PATHS, 12)
est_M10 = mc_estimate(M3[:, 10].astype(float))
est_cond = mc_estimate(M3[:, 11][S[:, 10] == 2].astype(float))
print(est_M10, "(exact 0)");  print(est_cond, "(exact 2^3 - 30*2 = -52)")
assert_close(est_M10, 0.0, k=4, name="E[S_10^3 - 30 S_10]")
assert_close(est_cond, 2.0**3 - 30 * 2, k=4, name="E[M_11 | S_10 = 2]")
bx, by, bse, bc = binned_conditional_mean(S[:, 10], M3[:, 11], MIN_BIN)
bmn = bx**3 - 30 * bx                                        # bin 안에서 M_10은 결정적
fig, ax = plt.subplots(); d = np.array([bmn.min() - 20, bmn.max() + 20])
ax.plot(d, d, "k--", lw=1, label="$y = x$")
ax.errorbar(bmn, by, yerr=1.96 * bse, fmt="o", color="C0", capsize=3, label="MC bin mean, 95% CI")
ax.set_xlabel("$M_{10} = S_{10}^3 - 30 S_{10}$"); ax.set_ylabel("$E[M_{11} \\mid S_{10}]$")
ax.set_title("E2. $S_n^3 - 3nS_n$: conditional means sit on the diagonal"); ax.legend(); plt.show()
```

</details>

### E3 가정을 깨보기

앞면 확률 $p=0.6$인 편향 동전으로 Z1, Z2를 다시 시뮬레이션하라(`random_walk_paths(..., p=0.6)`). 조건부 평균화 그림에서 무엇이 보이는가? 어떻게 고치면 마팅게일이 되는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$E[X]=0.6-0.4=0.2$, $\mathrm{Var}(X)=1-0.2^2=0.96$.

- **Z1**: $E[S_{n+1}\mid S_n=s]=s+0.2$ → 점들이 대각선 위로 0.2만큼 평행이동한다(서브마팅게일, 유리한 게임).
- **Z2**: $E[S_{n+1}^2-(n+1)\mid S_n=s]=s^2+2s(0.2)+1-(n+1)=(s^2-n)+0.4s$ → 편차가 $s$에 비례한다(기울기가 있어 서브도 슈퍼도 아니다).
- **수정**: 평균을 빼면 $\tilde S_n=S_n-0.2n$이 마팅게일이고, 분산으로 보정하면 $\tilde S_n^2-0.96n$도 마팅게일이다(Z2의 일반형 $S_n^2-n\sigma^2$를 중심화된 걸음에 적용).

```python
Sb = random_walk_paths(N_PATHS, 11, rng, p=0.6); n = np.arange(12)
mu, var = 0.6 - 0.4, 1 - (0.6 - 0.4)**2                          # 0.2, 0.96
bx, by, bse, bc = binned_conditional_mean(Sb[:, 10], Sb[:, 11], MIN_BIN)
print("Z1 deviation from diagonal:", (by - bx).round(3), " (exact 0.2)")
bx, by, bse, bc = binned_conditional_mean(Sb[:, 10], Sb[:, 11]**2 - 11, MIN_BIN)
print("Z2 deviation from diagonal:", (by - (bx**2 - 10)).round(2), " vs 0.4 s:", (0.4 * bx).round(2))
St = Sb - mu * n                                                   # 중심화된 걸음
assert_close(mc_estimate(St[:, 10]), 0.0, k=4, name="S_n - 0.2 n")
assert_close(mc_estimate(St[:, 10]**2 - var * 10), 0.0, k=4, name="(S_n - 0.2n)^2 - 0.96 n")
```

</details>

## 7. 정리

1. 조건부 기대 $E[X\mid\mathcal F_n]$은 현재 정보의 함수인 확률변수이며 탑 성질·아는 것 꺼내기·독립성 규칙으로 계산한다.
2. 마팅게일 $=$ 적응 $+$ 적분가능 $+$ $E[M_{n+1}\mid\mathcal F_n]=M_n$; 서브(≥)/슈퍼(≤)는 유리/불리한 게임.
3. 동물원: $S_n$, $S_n^2-n$, $e^{\theta S_n}\cosh^{-n}\theta$, $Z_n/m^n$, 폴리아 비율, 우도비, 평균 1 곱은 마팅게일; $S_n^2$는 드리프트 $+1$인 서브마팅게일이고 $S_n^2=(S_n^2-n)+n$이 둡 분해다.
4. 수치 검증법: 마르코프 상태로 bin을 나눠 $E[M_{n+1}\mid\text{상태}]$를 평균 내면 대각선에 놓여야 한다 — 비-예시는 대각선을 벗어난다. $E[M_n]=M_0$만으로는 판정할 수 없다.
5. 마팅게일은 고정 시각에서 $E[M_n]=M_0$을 보존한다. 무작위 시각(정지시각)에서의 보존은 조건이 필요하다 → 05b.

**Trap 카드**
- Q: 폴리아 항아리를 빨강 1·파랑 3으로 시작해 아주 많이 뽑으면 빨간 비율은 1/4 근처로 수렴하는가?
- A: 아니다. $E[X_n]=1/4$는 항상 성립하지만(마팅게일), 각 경로는 서로 다른 값으로 수렴한다 — 극한은 Beta(1,3) 확률변수다(05d). 초기 몇 번의 뽑기가 운명을 결정한다.

**다음 노트북: 05b** — 고정 시각의 기대값 보존 $E[M_n]=M_0$을 무작위 정지시각 $T$로 확장하는 선택적 정지 정리(optional stopping theorem)를 배우고, 도박꾼의 파산을 $S_n$, $S_n^2-n$, $(q/p)^{S_n}$으로 세 번째 방법으로 푼다. 조건이 깨지는 배가 전략(martingale betting)도 본다.

log/progress.md 한 줄 템플릿:
`- [ ] 05a 조건부 기대와 마팅게일 동물원 — YYYY-MM-DD, 예측 6개 중 __개 적중, 막힌 곳: ____, 다시 볼 것: 동물원 8후보의 한 줄 계산을 책 덮고 다시 쓰기`